In [1]:
import pickle
import pandas as pd

df = pd.read_csv('baf_complete.tsv', sep="\t")
df.columns = ["question", "entities", "mask", "answer_masked", "answer_unmasked"]

In [2]:
import json

with open("data/graph_structure.json", "r", encoding="utf-8") as f:
    graph_structure = json.load(f)

In [3]:
from openai import OpenAI
client = OpenAI()

def ask_chatgpt(question, model="gpt-5-mini"):
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": question}
        ]
    )
    return response.choices[0].message.content

In [4]:
gpt_inferences = []
count = 0
for masked_question in df['mask']:
    prompt = f'''You are an assistant specialized in reconstructing anonymized elements using structural graph context.
    The question you will see contains one placeholder: AD_HOC.
    The only available contextual information comes from graph_structure, which describes the relationships or topology relevant to the question.
    Your task: use graph_structure to infer the most plausible replacement for AD_HOC.
    Output strictly: return the inferred replacement text only — no commentary, formatting, or punctuation.

    Question: {masked_question} 
    
    graph_structure: {graph_structure}
    
    Reply only with your guess, do not explain.'''

    gpt_inference = ask_chatgpt(prompt)
    gpt_inferences.append(gpt_inference)
    count += 1
    print(count, gpt_inference)

1 Tom Hanks
2 Tom Hanks
3 Movie
4 Tom Hanks
5 Tom Hanks
6 Tom Hanks
7 Tom Hanks
8 Tom Hanks
9 Tom Hanks
10 Christopher Nolan
11 Director
12 Christopher Nolan
13 Christopher Nolan
14 Christopher Nolan
15 Christopher Nolan
16 Director
17 Christopher Nolan
18 Director
19 Christopher Nolan
20 Christopher Nolan
21 Inception
22 Christopher Nolan
23 Director
24 The Matrix
25 Movie
26 The Matrix
27 The Godfather
28 The Matrix
29 The Matrix
30 Inception
31 The Godfather
32 Movie
33 Movie
34 Christopher Nolan
35 Movie
36 Inception
37 Inception
38 Inception
39 Inception
40 Inception
41 name
42 The Godfather
43 The Godfather
44 Christopher Nolan
45 Drama
46 Genre
47 Inception
48 Inception
49 Movie
50 Drama
51 Drama
52 Movie
53 Drama
54 Movie
55 name
56 Action
57 Genre
58 The Matrix
59 Drama
60 Comedy
61 Movie
62 Inception
63 Inception
64 Drama
65 Movie
66 The Matrix
67 Movie
68 Movie
69 Movie
70 Movie
71 The Matrix
72 The Matrix
73 Movie
74 name
75 Inception
76 Movie
77 name
78 name
79 name
80 Mov

In [6]:
from typing import List
import re
def extract_bracketed_entities(sentence: str) -> List[str]:
    """
    Extract entities wrapped in square brackets [...] from the sentence.
    Returns a list of unique values (without brackets).
    """
    return list(re.findall(r'\[([^\[\]]+)\]', sentence))

In [8]:
sensitive_values_qa = []
for question in df['question']:
    sensitives = extract_bracketed_entities(question)
    sensitive_values_qa = sensitive_values_qa + sensitives

In [11]:
for guess, sensitive_value in zip(gpt_inferences, sensitive_values_qa[1:]):
    print(guess, ' : ' , sensitive_value)

Tom Hanks  :  Michelle Trachtenberg
Tom Hanks  :  Helen Mack
Movie  :  Shahid Kapoor
Tom Hanks  :  Paresh Rawal
Tom Hanks  :  Jeremy Piven
Tom Hanks  :  Chris O'Donnell
Tom Hanks  :  Darren McGavin
Tom Hanks  :  Martín Adjemián
Tom Hanks  :  John Pasquin
Christopher Nolan  :  Rowland Brown
Director  :  Aram Avakian
Christopher Nolan  :  Lloyd Ingraham
Christopher Nolan  :  Israel Horovitz
Christopher Nolan  :  Paul Wendkos
Christopher Nolan  :  Zack Snyder
Director  :  James B. Clark
Christopher Nolan  :  Robert Resnikoff
Director  :  Eddie Murphy
Christopher Nolan  :  Albert Zugsmith
Christopher Nolan  :  Garth Jennings
Inception  :  Joël Séria
Christopher Nolan  :  Lisa Langseth
Director  :  First Snow
The Matrix  :  Singin' in the Rain
Movie  :  War Horse
The Matrix  :  Puppet Master vs Demonic Toys
The Godfather  :  The Fourth Kind
The Matrix  :  Up in Arms
The Matrix  :  Pennies from Heaven
Inception  :  Make the Yuletide Gay
The Godfather  :  The Blind Sunflowers
Movie  :  The In

In [12]:
for guess, sensitive_value in zip(gpt_inferences, sensitive_values_qa[1:]):
    if guess == sensitive_value:
        print(guess)